# County-boundary cohort validation

Issue #27. Compare the existing eight-ZIP cohort, the Durham County polygon cohort, and their intersection using the same residential, price, feature, and date rules. All outputs are aggregates. The 2025 test split appears in counts only; no 2025 prices or predictions are inspected here. Run from the repository root with the supplied raw files.

The fixed Poisson loss was selected in the earlier ZIP-cohort validation. This notebook holds its configuration fixed and uses 2024 validation to compare geography choices. It does not approve a serving model or prediction interval.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import shapely
from homelens.data.boundary import load_county_boundary
from homelens.data.inventory import (
    REDFIN_COLUMNS,
    _file_identity,
    _normalized_zips,
    _read_csv,
    _require_columns,
)
from homelens.data.prepare import (
    NUMERIC_FEATURES,
    OUTPUT_COLUMNS,
    RESIDENTIAL_TYPES,
    STUDY_ZIPS,
    prepare_sales,
)
from homelens.modeling.baseline import (
    CATEGORICAL_COLUMNS,
    MODEL_CONFIG,
    SMALL_SLICE_MIN_ROWS,
    _features,
)
from sklearn.ensemble import HistGradientBoostingRegressor

raw_dir = Path("data/raw")
sales_path = raw_dir / "redfin_data.csv"
boundary_path = raw_dir / "durham_county_boundary.geojson"
sales = _read_csv(sales_path, ("ZIP OR POSTAL CODE", "MLS#"))
_require_columns(
    sales,
    sales_path,
    (*REDFIN_COLUMNS, "SALE TYPE", "STATE OR PROVINCE", "LATITUDE", "LONGITUDE"),
)
boundary = load_county_boundary(boundary_path)
official_zip, official_audit = prepare_sales(raw_dir)
print(
    {
        "source_rows": len(sales),
        "official_zip_rows": len(official_zip),
        "sales_sha256": _file_identity(sales_path)["sha256"],
        "boundary_sha256": _file_identity(boundary_path)["sha256"],
    }
)

{'source_rows': 23545, 'official_zip_rows': 17279, 'sales_sha256': '01d4cb47bd23346f15dce22b32485cd2c2318fe11d70331152dfccdd6ec5aa96', 'boundary_sha256': '96130b661255edbbe0229209eed8f4fb0661035737c0c7bae34eaedfa06ca789'}


## Shared quality filters

Apply the preparation rules before either geography rule. A final equality check below confirms that the reconstructed ZIP cohort matches the existing preparation command.

In [2]:
quality = sales.drop_duplicates().copy()
excluded = {"exact_duplicates": len(sales) - len(quality)}


def retain(frame, mask, reason):
    kept = frame.loc[mask.fillna(False)].copy()
    excluded[reason] = len(frame) - len(kept)
    return kept


quality = retain(
    quality,
    quality["SALE TYPE"].astype("string").str.strip().eq("PAST SALE"),
    "not_past_sale",
)
dates = pd.to_datetime(quality["SOLD DATE"], format="mixed", errors="coerce")
quality = retain(quality, dates.notna(), "missing_or_invalid_sale_date")
quality["sale_date"] = dates.loc[quality.index].dt.strftime("%Y-%m-%d")
prices = pd.to_numeric(quality["PRICE"], errors="coerce")
quality = retain(
    quality,
    prices.gt(0) & prices.lt(float("inf")),
    "missing_invalid_or_nonpositive_price",
)
quality["price_usd"] = prices.loc[quality.index]
quality = retain(
    quality,
    quality["PROPERTY TYPE"].isin(RESIDENTIAL_TYPES),
    "outside_initial_residential_types",
)
quality = retain(
    quality,
    quality["STATE OR PROVINCE"].astype("string").str.strip().eq("NC"),
    "outside_north_carolina",
)
for column in NUMERIC_FEATURES:
    quality[column] = pd.to_numeric(quality[column], errors="coerce")
quality["zip"] = _normalized_zips(quality["ZIP OR POSTAL CODE"])
complete = quality[list(NUMERIC_FEATURES)].notna().all(axis=1) & quality["zip"].notna()
quality = retain(quality, complete, "missing_core_features")
quality = retain(quality, quality["zip"].str.fullmatch(r"\d{5}"), "invalid_zip_format")
sale_year = pd.to_datetime(quality["sale_date"]).dt.year
plausible = (
    quality["BEDS"].ge(0)
    & quality["BEDS"].lt(float("inf"))
    & quality["BATHS"].gt(0)
    & quality["BATHS"].lt(float("inf"))
    & quality["SQUARE FEET"].ge(300)
    & quality["SQUARE FEET"].lt(float("inf"))
    & quality["YEAR BUILT"].ge(1800)
    & quality["YEAR BUILT"].le(sale_year)
    & quality["YEAR BUILT"].mod(1).eq(0)
)
quality = retain(quality, plausible, "implausible_core_features")
print(
    pd.DataFrame(
        [{"stage": key, "excluded_rows": value} for key, value in excluded.items()]
    ).to_string(index=False)
)
print("shared_quality_rows:", len(quality))

                               stage  excluded_rows
                    exact_duplicates           1416
                       not_past_sale              1
        missing_or_invalid_sale_date           3728
missing_invalid_or_nonpositive_price              0
   outside_initial_residential_types           1006
              outside_north_carolina              0
               missing_core_features             49
                  invalid_zip_format              0
           implausible_core_features             15
shared_quality_rows: 17330


## Geography and split counts

Keep source row indices only in memory to compare the same sale across cohorts. County inclusion uses polygon coverage, including boundary points. A ZIP remains a model predictor in both cohorts, so rows with an absent or malformed ZIP fail the shared quality filter.

In [3]:
lat = pd.to_numeric(quality["LATITUDE"], errors="coerce")
lon = pd.to_numeric(quality["LONGITUDE"], errors="coerce")
usable = lat.between(33, 37) & lon.between(-85, -75)
in_county = pd.Series(False, index=quality.index)
points = shapely.points(
    lon.loc[usable].to_numpy(dtype=float), lat.loc[usable].to_numpy(dtype=float)
)
in_county.loc[usable] = shapely.covers(boundary, points)
quality["in_county"] = in_county
quality["study_zip"] = quality["zip"].isin(STUDY_ZIPS)


def model_rows(frame):
    result = frame.rename(
        columns={
            "BEDS": "beds",
            "BATHS": "baths",
            "SQUARE FEET": "square_feet",
            "YEAR BUILT": "year_built",
            "PROPERTY TYPE": "property_type",
        }
    ).copy()
    result["year_built"] = result["year_built"].astype(int)
    result["_source_row"] = result.index
    result["split"] = "train"
    result.loc[result["sale_date"].ge("2024-01-01"), "split"] = "validation"
    result.loc[result["sale_date"].ge("2025-01-01"), "split"] = "test"
    return result[[*OUTPUT_COLUMNS, "_source_row", "in_county", "study_zip"]]


zip_cohort = model_rows(quality.loc[quality["study_zip"]])
county_cohort = model_rows(quality.loc[quality["in_county"]])
verified_zip_cohort = model_rows(
    quality.loc[quality["study_zip"] & quality["in_county"]]
)
reconstructed = (
    zip_cohort[list(OUTPUT_COLUMNS)]
    .sort_values(["sale_date", "zip", "price_usd"], kind="stable")
    .reset_index(drop=True)
)
pd.testing.assert_frame_equal(reconstructed, official_zip, check_dtype=False)
assert len(zip_cohort) == official_audit["prepared_rows"]
split_counts = (
    pd.DataFrame(
        {
            "ZIP cohort": zip_cohort["split"].value_counts(),
            "County cohort": county_cohort["split"].value_counts(),
            "County-verified ZIP": verified_zip_cohort["split"].value_counts(),
        }
    )
    .reindex(["train", "validation", "test"])
    .fillna(0)
    .astype(int)
)
print(split_counts.to_string())
print(
    {
        "unusable_coordinate_rows_after_quality": int((~usable).sum()),
        "common_rows": len(
            set(zip_cohort["_source_row"]) & set(county_cohort["_source_row"])
        ),
        "zip_only_rows": int((~zip_cohort["in_county"]).sum()),
        "county_only_rows": int((~county_cohort["study_zip"]).sum()),
    }
)

            ZIP cohort  County cohort  County-verified ZIP
split                                                     
train            13280          13138                13102
validation        2979           2936                 2928
test              1020           1013                 1010
{'unusable_coordinate_rows_after_quality': 0, 'common_rows': 17040, 'zip_only_rows': 239, 'county_only_rows': 47}


In [4]:
print("Inside-county sales with non-study ZIP labels:")
print(
    county_cohort.loc[~county_cohort["study_zip"]]
    .groupby("zip")
    .size()
    .sort_values(ascending=False)
    .to_string()
)
print("Outside-county sales with study ZIP labels:")
print(
    zip_cohort.loc[~zip_cohort["in_county"]]
    .groupby("zip")
    .size()
    .sort_values(ascending=False)
    .to_string()
)
print(
    "Geography/label conflicts for review; non-study ZIPs are not necessarily errors."
)

Inside-county sales with non-study ZIP labels:
zip
27517    20
27560     3
27572     3
27278     2
22703     2
27709     2
27131     1
17704     1
27113     1
27519     1
27514     1
27523     1
27377     1
27571     1
27604     1
27613     1
27702     1
27714     1
27716     1
28211     1
37704     1
Outside-county sales with study ZIP labels:
zip
27705    120
27703    109
27712      7
27713      2
27707      1
Geography/label conflicts for review; non-study ZIPs are not necessarily errors.


## Fixed 2024 model comparison

Fit the same preselected Poisson gradient model on each cohort's pre-2024 training rows. Assess 2024 validation only. Full-cohort metrics describe different sales; common-sale metrics compare predictions on the same homes. Slices with fewer than 30 sales show counts without error estimates.

In [5]:
def fit_and_score(frame):
    work = frame.loc[frame["split"].isin(["train", "validation"])].copy()
    assert not work["split"].eq("test").any()
    train = work.loc[work["split"].eq("train")]
    validation = work.loc[work["split"].eq("validation")]
    categories = {
        column: sorted(train[column].astype("string").unique().tolist())
        for column in CATEGORICAL_COLUMNS
    }
    model = HistGradientBoostingRegressor(**{**MODEL_CONFIG, "loss": "poisson"})
    model.fit(_features(train, categories), train["price_usd"])
    scored = validation[
        ["_source_row", "sale_date", "price_usd", "zip", "in_county", "study_zip"]
    ].copy()
    scored["prediction"] = model.predict(_features(validation, categories))
    scored["absolute_error"] = (scored["prediction"] - scored["price_usd"]).abs()
    scored["signed_error"] = scored["prediction"] - scored["price_usd"]
    scored["half"] = np.where(
        scored["sale_date"].lt("2024-07-01"), "Jan-Jun", "Jul-Dec"
    )
    return scored


def metrics(frame):
    count = len(frame)
    if count < SMALL_SLICE_MIN_ROWS:
        return {"rows": count, "mae_usd": None, "mean_signed_error_usd": None}
    return {
        "rows": count,
        "mae_usd": round(float(frame["absolute_error"].mean())),
        "mean_signed_error_usd": round(float(frame["signed_error"].mean())),
    }


zip_scored = fit_and_score(zip_cohort)
county_scored = fit_and_score(county_cohort)
verified_zip_scored = fit_and_score(verified_zip_cohort)
common_validation = set(verified_zip_scored["_source_row"])
comparison = pd.DataFrame(
    [
        {
            "cohort": name,
            "scope": scope,
            **metrics(
                scored.loc[scored["_source_row"].isin(common_validation)]
                if scope == "common validation"
                else scored
            ),
        }
        for name, scored in [
            ("ZIP", zip_scored),
            ("County", county_scored),
            ("County-verified ZIP", verified_zip_scored),
        ]
        for scope in ("all validation", "common validation")
    ]
)
print(comparison.to_string(index=False))
for name, scored, column in [
    ("ZIP", zip_scored, "in_county"),
    ("County", county_scored, "study_zip"),
    ("County-verified ZIP", verified_zip_scored, "zip"),
]:
    print(name, "2024 halves:")
    print(
        pd.DataFrame(
            [
                {"half": half, **metrics(part)}
                for half, part in scored.groupby("half", sort=True)
            ]
        ).to_string(index=False)
    )
    print(name, "geography slices:")
    print(
        pd.DataFrame(
            [
                {column: str(value), **metrics(part)}
                for value, part in scored.groupby(column, sort=True)
            ]
        ).to_string(index=False)
    )

             cohort             scope  rows  mae_usd  mean_signed_error_usd
                ZIP    all validation  2979    74463                 -47513
                ZIP common validation  2928    74761                 -47678
             County    all validation  2936    75394                 -48831
             County common validation  2928    75100                 -48464
County-verified ZIP    all validation  2928    74862                 -48895
County-verified ZIP common validation  2928    74862                 -48895
ZIP 2024 halves:
   half  rows  mae_usd  mean_signed_error_usd
Jan-Jun  1548    72653                 -46194
Jul-Dec  1431    76421                 -48940
ZIP geography slices:
in_county  rows  mae_usd  mean_signed_error_usd
    False    51    57342                 -38025
     True  2928    74761                 -47678
County 2024 halves:
   half  rows  mae_usd  mean_signed_error_usd
Jan-Jun  1519    73375                 -46698
Jul-Dec  1417    77559             

## Interpretation

The reconstructed 17,279-row ZIP cohort exactly matches the existing preparation output. After the shared quality rules, 17,330 sales remain. The county polygon retains 17,087; 17,040 are also in the study ZIPs. Thus 239 ZIP-cohort sales lie outside the county, while 47 inside-county sales carry other ZIP labels. All quality-eligible rows have usable coordinates.

On the same 2,928 sales in 2024 validation, the fixed Poisson model has MAE of USD 74,761 with the original ZIP training cohort, USD 75,100 with the full county cohort, and USD 74,862 with the county-verified study-ZIP cohort. The USD 101 difference between the original and county-verified ZIP models is small relative to either MAE. The full county candidate adds only 8 non-study-ZIP validation sales, below the 30-row reporting threshold. Several of its ZIP labels are geographically surprising; this notebook does not change or certify them.

Select the county-verified eight-ZIP subset for the next reproducible preparation and evaluation procedure. This choice removes known outside-county sales while keeping a supported categorical ZIP feature. It covers a verified subset of Durham County, not the whole county. Keep the broader county candidate diagnostic until its sparse ZIP labels are checked with authoritative ZIP geography. The same 2024 validation period helped select the earlier Poisson loss, so this comparison is exploratory. The 2025 holdout remains reserved for one final descriptive evaluation in Python.